### V2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import re
import plotly
import plotly.express as px
import pickle

print(f"numpy=={np.__version__}")
print(f"pandas=={pd.__version__}")
print(f"matplotlib=={matplotlib.__version__}")
print(f"seaborn=={sns.__version__}")
print(f"re=={re.__version__}")
print(f"plotly=={plotly.__version__}")

df = pd.read_csv(filepath_or_buffer = "data/extracted/extraction_2024-04-17_spain.csv")
df_scrape = pd.read_csv(filepath_or_buffer = "data/extracted/extraction_2024-04-17_spain2.csv")

print(df.shape)
print(df_scrape.shape)

df.head(3)

In [ ]:
df_scrape.columns

In [ ]:
pd.merge(left = df, right = df_scrape, on = ["job_id", "description"]).shape

In [ ]:
df = pd.concat([df, df_scrape], ignore_index = True)

df = df.drop_duplicates("job_id")

### Funciones: General

In [ ]:
def clean_regular_brackets(string):

    pattern = r"\([^()]*\)"

    string = re.sub(pattern, "", string)

    return string.strip()

### Funciones: Location

In [ ]:
def get_location_spain(string):
    """
    string: Cadena que contiene la ubicación de la oferta de empleo.
    La función limpia espacios en blanco, elimina parentesis y unicodes (\\u200e).
    Si string contiene "Madrid" o "Anywhere" retorna "Madrid" o "Anywhere" respectivamente.
    Si string es solamente "Spain" retorna "Spain".
    ---------
    return: Retorna información extra de la ubicación, suponiendo que todos los empleos son de España o "Anywhere".
    """
    
    if type(string) == str:
        
        if string == "Anywhere" or string == "Spain":
            return string
        
        if "Madrid" in string:
            return "Madrid"
        
        # Elimina Paréntesis
        string = clean_regular_brackets(string)
        
        # Elimina Unicode \u2000e
        string = string.replace("\u200e", "")
    
        string = string.strip()
        
        # Si "," y "Spain" están en string aplica .split()
        if ("," in string) & ("Spain" in string):
            
            string = string.split(",")[0]
    
        return string
        
    return np.nan

In [ ]:
# get_location_spain?

In [ ]:
# Transformamos Location
df["location"] = df["location"].apply(lambda x : get_location_spain(x))

In [ ]:
# Visualización en Plotly de "location"

df_location = pd.DataFrame(df["location"].value_counts()).reset_index()

# Muestra los 10 lugares más comunes
fig = px.bar(data_frame = df_location.iloc[:10, :],
             x = "location",
             y = "count")

fig.show()

### Funciones: Via

In [ ]:
def get_source(string):
    
    """
    string: Cadena que contiene la fuente de la oferta de empleo.
    
    Elimina la cadena "via " de string.
    """
    
    if type(string) == str:
        
        string = string.replace("via ", "")
        
        return string
            
            
    return np.nan

In [ ]:
# Transformamos Via
df["via"] = df["via"].apply(lambda x : get_source(x))

In [ ]:
# Visualización en Plotly de "via"

df_via = pd.DataFrame(df["via"].value_counts()).reset_index()

# Muestra las 10 fuentes más comunes
fig = px.bar(data_frame = df_via.iloc[:10, :],
             x = "via",
             y = "count")

fig.show()

In [ ]:
df[df["description"]== df[df.duplicated("description")].iloc[-1, 4]]

In [ ]:
%%time

import pickle
import os

data = list()


for file_name in os.listdir("json_searches_spain"):

    with open(file = f"json_searches_spain/{file_name}", mode = "br") as file:
        j = pickle.load(file)

    if "jobs_results" in j.keys():
        data.extend(j["jobs_results"])

In [ ]:
pd.json_normalize(data)